# Notebook 06 — NumPy Linear Algebra

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 06 of 20  
**Prerequisites:** Notebook 05  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Principal Component Analysis (PCA) is the single most-used dimensionality reduction
technique in genomics. It is built on three linear algebra operations: matrix
multiplication, covariance matrices, and eigendecomposition. Understanding those
operations means you can diagnose when PCA misbehaves — not just run `PCA.fit_transform()`
and trust the result.

---
## Step 2 — Intuition

A covariance matrix $C$ tells you how each gene's expression varies with every other
gene's expression. Its eigenvectors are the directions of maximum variance — the
principal components. The eigenvalue tells you *how much* variance lies in that
direction.

Projecting data onto the top-2 eigenvectors compresses a 20,000-dimensional
problem into 2 dimensions while keeping the most information.

---
## Step 3 — Biological Background

**PCA on gene expression data:**  
- Each sample is a point in $\mathbb{R}^{n_{\text{genes}}}$ space.  
- PCA finds directions of greatest variance among samples.  
- PC1 often separates biological conditions (e.g., tumour vs normal).  
- PC2 often captures a secondary effect (e.g., batch, sex).  
- If PC1 explains batch instead of biology, normalisation failed — this is detectable
  by colouring the PCA plot by batch label.

Singular Value Decomposition (SVD) is mathematically identical to PCA but numerically
more stable for large sparse matrices. Most production tools use SVD internally.

---
## Step 4 — Mathematical Explanation

Let $X \in \mathbb{R}^{n \times p}$ be the centered data matrix ($n$ samples, $p$ genes).  

**Covariance matrix:**
$$C = \frac{1}{n-1} X^T X \quad \in \mathbb{R}^{p \times p}$$

**Eigendecomposition:** $C = V \Lambda V^T$, where  
- $V$: matrix of eigenvectors (principal components)  
- $\Lambda$: diagonal matrix of eigenvalues (variance explained)  

**Projection:** $Z = X V_k$ where $V_k$ contains the top-$k$ eigenvectors.

**SVD route:** $X = U \Sigma V^T$, then $Z = U_k \Sigma_k$. Equivalent to PCA
when $X$ is centered.

---
## Step 5 — Computational Explanation

| Operation | NumPy API | Notes |
|-----------|-----------|-------|
| Matrix multiply | `A @ B` or `np.matmul` | Prefer `@` operator |
| Transpose | `A.T` | No copy for C-contiguous arrays |
| Covariance | `np.cov(X.T)` | Bessel-corrected (divides by n-1) |
| Eigendecomposition | `np.linalg.eigh(C)` | For symmetric matrices; returns sorted eigenvalues |
| SVD | `np.linalg.svd(X, full_matrices=False)` | Returns U, s, Vt |
| Determinant | `np.linalg.det(A)` | Often better via log-determinant |
| Solve linear system | `np.linalg.solve(A, b)` | Prefer over explicit inverse |

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
rng = np.random.default_rng(seed=42)

In [ ]:
# Cell 6.1 — Matrix multiplication: dot products and gene co-expression
# Gene co-expression: how correlated are gene pairs across samples?
# (simple inner products, not Pearson r — but same idea)

X = rng.standard_normal((5, 3))  # 5 genes, 3 samples
print(f"X shape: {X.shape}")

# Gene-gene inner product matrix
G = X @ X.T  # shape (5, 5)
print(f"G = X @ X.T   shape: {G.shape}")

# Sample-sample inner product
S = X.T @ X  # shape (3, 3)
print(f"S = X.T @ X   shape: {S.shape}")

print(f"\nG[0,0] == sum(X[0]**2): {np.isclose(G[0,0], (X[0]**2).sum())}")

In [ ]:
# Cell 6.2 — PCA from scratch using eigendecomposition
def pca_eig(X: np.ndarray, n_components: int = 2) -> tuple[np.ndarray, np.ndarray]:
    """
    PCA via eigendecomposition of the covariance matrix.

    Parameters
    ----------
    X : array of shape (n_samples, n_features)
    n_components : number of principal components to return

    Returns
    -------
    Z : projection, shape (n_samples, n_components)
    explained_variance_ratio : shape (n_components,)
    """
    # Step 1: center
    Xc = X - X.mean(axis=0)
    # Step 2: covariance matrix
    C = np.cov(Xc.T)           # shape (p, p), Bessel-corrected
    # Step 3: eigendecomposition (eigh for symmetric matrices)
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    # Step 4: sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    # Step 5: project
    Z = Xc @ eigenvectors[:, :n_components]
    evr = eigenvalues[:n_components] / eigenvalues.sum()
    return Z, evr

# Generate synthetic data: 2 clusters in 50-dimensional space
cluster1 = rng.standard_normal((30, 50)) + np.array([2.0] * 50)
cluster2 = rng.standard_normal((30, 50)) - np.array([2.0] * 50)
X_data = np.vstack([cluster1, cluster2])  # shape (60, 50)
labels = np.array([0]*30 + [1]*30)

Z, evr = pca_eig(X_data, n_components=2)
print(f"Projection shape: {Z.shape}")
print(f"Explained variance ratio: PC1={evr[0]:.3f}, PC2={evr[1]:.3f}")

In [ ]:
# Cell 6.3 — PCA via SVD (numerically preferred)
def pca_svd(X: np.ndarray, n_components: int = 2) -> tuple[np.ndarray, np.ndarray]:
    """PCA via SVD — numerically stable for large/sparse matrices."""
    Xc = X - X.mean(axis=0)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    Z = U[:, :n_components] * s[:n_components]
    variance = (s**2) / (X.shape[0] - 1)
    evr = variance[:n_components] / variance.sum()
    return Z, evr

Z_svd, evr_svd = pca_svd(X_data, n_components=2)

# The projections should be the same (up to sign flip)
print(f"SVD EVR: PC1={evr_svd[0]:.3f}, PC2={evr_svd[1]:.3f}")
print(f"Projections agree (abs): {np.allclose(np.abs(Z), np.abs(Z_svd), atol=1e-8)}")

In [ ]:
# Cell 6.4 — Validating against scikit-learn PCA
from sklearn.decomposition import PCA

sk_pca = PCA(n_components=2)
Z_sk = sk_pca.fit_transform(X_data)

print("Explained variance ratio (sklearn):", sk_pca.explained_variance_ratio_.round(3))
print(f"Our SVD EVR match: {np.allclose(evr_svd, sk_pca.explained_variance_ratio_, atol=1e-6)}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — PCA scatter plot coloured by cluster label
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, Z_plot, title in zip(axes,
                             [Z_svd, Z_sk],
                             ["PCA (SVD, from scratch)", "PCA (scikit-learn)"]):
    for label, color, name in [(0, "steelblue", "Cluster 1"), (1, "tomato", "Cluster 2")]:
        mask = labels == label
        ax.scatter(Z_plot[mask, 0], Z_plot[mask, 1],
                   c=color, alpha=0.7, label=name, edgecolors="none")
    ax.set_xlabel(f"PC1 ({evr_svd[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({evr_svd[1]:.1%} variance)")
    ax.set_title(title)
    ax.legend()

plt.suptitle("PCA separates two synthetic clusters", fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Add `pca_svd(X, n_components)` to `compbio_utils/stats.py` with full type hints and docstring.
2. Compute the Pearson correlation matrix of `X_data` using `np.corrcoef`. How does it
   differ from the covariance matrix?
3. The SVD of an $m \times n$ matrix returns shapes $U (m \times m)$, $s (\min(m,n))$,
   $V^T (n \times n)$. What does `full_matrices=False` change, and why do we prefer it?
4. Implement a 'scree plot' — a bar chart of explained variance ratio per PC. Add it to
   `compbio_utils/plotting.py` as `scree_plot(evr)`.

---
## Quiz — Active Recall

1. What does the eigenvalue of a principal component represent biologically?
2. Why should you center the data matrix before computing PCA?
3. `np.linalg.eigh` vs `np.linalg.eig` — when do you use each?
4. SVD gives $X = U\Sigma V^T$. What are the shapes of $U$, $\Sigma$, $V^T$ for
   an $n \times p$ matrix when `full_matrices=False`?
5. If PC1 explains 80% of variance but separates by batch rather than condition,
   what does that tell you about the data?

---
## Reflection

**Date completed:** ____________________

> *[Could you derive the eigendecomposition route on paper? Is pca_svd in compbio_utils? What was the most confusing part of SVD?]*

---
*Next: `07_random_numbers_reproducibility.ipynb`*